In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from itertools import combinations
from tqdm.notebook import tqdm
from pathlib import Path

In [2]:
df = pd.read_csv('../data/clinical/data_June_2025.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2140 entries, 0 to 2139
Columns: 4431 entries, PTID to AB_SUVR_GM_GLOBL
dtypes: float64(4203), int64(69), object(159)
memory usage: 72.3+ MB


/tmp/ipykernel_528815/3661735777.py:1: DtypeWarning: Columns (0,20,22,34,36,38,42,48,52,66,68,70,75,86,94,603,612,630,631,632,633,634,635,636,637,638,639,640,641,642,643,644,731,740,751,754,764,796,978,982,984,1006,1008,1017,1028,1030,1077,1247,1283,1299,1316,1388,1399,1432,1441,1609,1611,1613,1615,1617,1621,1625,1629,1641,1645,1988,1990,1992,1994,2184,2186,2372,2374,2393,2394,2396,2408,2409,2410,2440,2445,2450,2454,2519,2520,2661,2662,2663,2694,2705,3076,3100,3314,3316,3324,3326,3404,3407,3419,3434,3437,3449,3452,3456,3458,3594,3596,3601,3642,3674,4156,4158) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/clinical/data_June_2025.csv')


In [3]:
df["PTID"]

0        110001
1        110002
2        110003
3        110003
4        110003
         ...   
2135    A16-003
2136    A16-004
2137    A16-006
2138    A16-007
2139    A17-001
Name: PTID, Length: 2140, dtype: object

In [9]:
df.dropna(subset=["VOL_HIPP_L", "PTAU_217_CONCNTRTN", "LASSI_A_CR2"])["FL_UDSD"].value_counts()

FL_UDSD
4.0    157
3.0     96
1.0     51
5.0     51
6.0     47
2.0     20
Name: count, dtype: int64

# 1.Pre-processing

In [4]:
df = df[~df['PTID'].str.startswith('A', na=False)]

## MMSE Processing
There are two MMSE columns and I would noticed that some of 'FL_MMSE' had a value of 999,-4, or empty so i just use the column 'MMSE' as a fall back to fill them.

Filling out MMSE Values. Using FL_MMSE Column as is the most complete, if the value is 999 or -4 set it to NaN. 
fallback to MMSE values if is missing.

In [5]:
df["COMBINED_MMSE"] = df["FL_MMSE"].replace([999, -4], np.nan)
df["COMBINED_MMSE"] = df["COMBINED_MMSE"].fillna(df["MMSE"])
df = df.drop(columns=["FL_MMSE", "MMSE"])

In [6]:
df = df.rename(columns={"COMBINED_NE4S": "APOE4S", "COMBINED_MMSE": "MMSE"})

In [7]:
FEATURES=["PTID","VISITYR", "CDRSUM","CDRGLOB","MMSE","HVLT_DR","LASSI_A_CR2","LASSI_B_CR1","LASSI_B_CR2","APOE4S","PTAU_217_CONCNTRTN", "AMYLPET","FL_UDSD", "NACCETPR"]

In [8]:
df_filter = df[FEATURES]

In [9]:
df_filter.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2134 entries, 0 to 2133
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   PTID                2134 non-null   object 
 1   VISITYR             2134 non-null   int64  
 2   CDRSUM              2134 non-null   float64
 3   CDRGLOB             2134 non-null   float64
 4   MMSE                2019 non-null   float64
 5   HVLT_DR             1849 non-null   float64
 6   LASSI_A_CR2         1830 non-null   float64
 7   LASSI_B_CR1         1826 non-null   float64
 8   LASSI_B_CR2         1810 non-null   float64
 9   APOE4S              1903 non-null   float64
 10  PTAU_217_CONCNTRTN  672 non-null    float64
 11  AMYLPET             1430 non-null   float64
 12  FL_UDSD             2120 non-null   float64
 13  NACCETPR            1611 non-null   float64
dtypes: float64(12), int64(1), object(1)
memory usage: 250.1+ KB


In [10]:
# drop_df_filter = df_filter.dropna(subset=["CDRGLOB", "AMYLPET","FL_UDSD", "NACCETPR"])

In [11]:
# drop_df_filter.info()

In [12]:
df_filter.to_csv('../data/clinical_data/clinical_preprocessed.csv', index=False)

OSError: Cannot save file into a non-existent directory: '../data/clinical_data'